# Export `laya-typed-decisions` for the browser

Produces the two ONNX graphs this app loads — a weight-only 8-bit encoder and an fp16-storage head — from [`convaiinnovations/laya-typed-decisions`](https://huggingface.co/convaiinnovations/laya-typed-decisions), and pushes them to the Hub.

**Why this checkpoint.** On typed decisions the base checkpoints score 0.362 (English) and 0.342 (multilingual) against a 0.461 majority-class baseline — below the trivial answer. This fine-tuned one scores **0.766**. Nothing else on the accuracy list comes close, and there is no ONNX build of it yet.

**Runtime.** A free Colab CPU or T4 runtime is enough: ~12.7 GB RAM against the 6–8 GB this needs, and about 6 GB of disk. High-RAM is not required and neither is the GPU — this is graph tracing and weight quantization, both CPU-bound. Expect 20–40 minutes, most of it the two downloads.

**The one real footgun** is in the next cell's pin: `transformers>=5.0`. Version 4.x misreads this encoder's split `rope_theta` — 160000 on the 10 full-attention layers, 10000 on the 18 sliding-attention ones — and does it **without raising**, so the export succeeds and the model is quietly wrong. The verification gate at the end is what catches that, and it is not optional.

In [ ]:
# 1. Refuse early rather than dying halfway through a 40-minute job.
import shutil, sys, psutil

ram = psutil.virtual_memory().total / 1e9
disk = shutil.disk_usage('/content').free / 1e9
print(f'RAM {ram:.1f} GB   free disk {disk:.1f} GB   python {sys.version.split()[0]}')
assert ram >= 9, f'need ~8 GB to trace a 421M model; this runtime has {ram:.1f} GB'
assert disk >= 12, f'need ~6 GB for the checkpoint and the graphs, with room to spare; have {disk:.1f} GB'
print('ok')

In [ ]:
# 2. Dependencies. The transformers pin is load-bearing -- see the note above.
%pip install -q --upgrade 'transformers>=5.0' safetensors onnx 'onnxruntime>=1.20' huggingface_hub
import transformers, onnx, onnxruntime, torch
print('transformers', transformers.__version__, '| onnx', onnx.__version__,
      '| onnxruntime', onnxruntime.__version__, '| torch', torch.__version__)
major = int(transformers.__version__.split('.')[0])
assert major >= 5, 'transformers 4.x misreads this encoder rope config silently; restart the runtime after the install'

In [ ]:
# 3. The export pipeline, and the checkpoint under the directory name its scripts expect.
!git clone --depth 1 -q https://github.com/nvkudva/laya-web.git /content/laya-web
%cd /content/laya-web/export

from huggingface_hub import snapshot_download
snapshot_download('convaiinnovations/laya-typed-decisions', local_dir='laya-en')
!ls -la laya-en && cat laya-en/rl_agent_config.json

In [ ]:
# 4. Reference outputs from the PyTorch model, so the gate at the end has something
#    to compare against. Skipping this makes every later 'PASS' meaningless.
!python make_fixtures.py && python dump_golden.py

In [ ]:
# 5. The recipe, exactly as nvkudva documents it for the English build.
#
#    Weight-only, deliberately: plain dynamic int8 collapses this architecture to 69%
#    argmax agreement. The damage is activation quantization, not weight precision --
#    ModernBERT has outlier activation channels a per-tensor dynamic scale cannot
#    represent. Leaving activations in fp32 avoids it, and is also what makes the
#    result batch-exact.
!python export_onnx.py
!python quantize_nbits.py --block-size 64 --out out/enc_nb8_bs64.onnx
!python embed_fp16.py --src out/enc_nb8_bs64.onnx --out out/encoder_q8.onnx
!python head_fp16_storage.py
!ls -la out/

In [ ]:
# 6. The gate. If this does not say PASS, nothing below should be uploaded --
#    a quantized model that answers differently is not a smaller model, it is
#    another one wearing the same name.
!python verify_onnx.py --encoder out/encoder_q8.onnx --head out/head_q8.onnx --gate

In [ ]:
# 7. Push. Needs a Hub token with write access; paste it when prompted.
#    REPO is where the app will fetch from -- change the owner to your account.
REPO = 'alfred361/laya-typed-decisions-web-q8'

from huggingface_hub import HfApi, notebook_login
notebook_login()
api = HfApi()
api.create_repo(REPO, repo_type='model', exist_ok=True)

import shutil, os
os.makedirs('upload/v1', exist_ok=True)
for f in ['out/encoder_q8.onnx', 'out/encoder_q8.onnx.data', 'out/head_q8.onnx', 'out/head_q8.onnx.data']:
    if os.path.exists(f):
        shutil.copy(f, 'upload/v1/')
# The app reads these three from the same directory as the graphs.
shutil.copy('laya-en/rl_agent_config.json', 'upload/v1/')
shutil.copy('laya-en/tokenizer/tokenizer.json', 'upload/v1/')
shutil.copy('laya-en/tokenizer/tokenizer_config.json', 'upload/v1/')

# v1 is part of the browser cache key on purpose: caches key on URL, so a re-export
# goes to v2 rather than serving stale weights to anyone who already has v1.
api.upload_folder(folder_path='upload', repo_id=REPO, repo_type='model')
print('https://huggingface.co/' + REPO)

## Then

Send the repository id back. Wiring it into the app is a new entry in `src/models.ts` — `layout: "split"`, `tokenizerPath: ""`, `backend: "wasm"`, its own `cache` bucket — and nothing else, because it is the same architecture and the same tokenizer as the English build already running there.

Two things worth knowing before it is treated as the accurate option:

- It is **English only**, like the base English checkpoint, and fails the same way on non-Latin script.
- Its 0.766 was measured on the four workflows it was fine-tuned for — invoice processing, security incidents, customer service, agent-trace observability. On questions shaped like something else it is a general checkpoint again. Worth measuring on your own questions rather than inheriting the headline.